# Pydantic

In this notebook we will look at the the Pydantic library
- BaseModel
- Field
- Validator
- Instructor

In [12]:
from dataclasses import dataclass

@dataclass
class Person:
	name: str
	age: int

print(Person(name="Sam", age="10"))

#error
Person(name="Sam", age="10").age + 1

Person(name='Sam', age='10')


TypeError: can only concatenate str (not "int") to str

BaseModel

In [13]:
from pydantic import BaseModel

class Person(BaseModel):
	name: str
	age: int

print(Person(name="Sam", age="10"))

Person(name="Sam", age="10").age + 1

name='Sam' age=10


11

Field function

In [40]:
from pydantic import BaseModel, Field

class User(BaseModel):
    id: int
    name: str = Field(..., max_length=50)
    email: str = Field(..., pattern=r'^\S+@\S+\.\S+$')

Validator

In [36]:
from pydantic import BaseModel, field_validator

class User(BaseModel):
    id: int
    name: str
    email: str

    @field_validator('email')
    def validate_email(cls, value):
        if '@' not in value:
            raise ValueError('Invalid email address')
        return value

LLM Generation

LLM libraries compatibility :
- OpenAI
- Ollama
- llama-cpp-python
- Anthropic
- Gemini
- Grok

In [6]:
from openai import OpenAI
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
import instructor

# First you define a pydantic model
class User(BaseModel):
    name: str
    age: int
    children: int | None
    fact: List[str] = Field(..., description="A list of facts about the character")

    @field_validator("name")
    def name_must_be_uppercase(cls, v: str):
        if v.upper() != v:
            raise ValueError("Name must be uppercase, please fix.")
        return v

# Then Patch the client you want to use
client = instructor.from_openai(
    OpenAI(
        base_url="",
        api_key="ollama",  # required, but unused
    ),
    # For Ollama you need to specify the mode argument
    mode=instructor.Mode.JSON,
)

# Function to call the API and print the result
def call_api():
    resp = client.chat.completions.create(
        model="llama3.1",
        messages=[
            {
                "role": "user",
                "content": "Imagine a japanese doctor and tell me about him. Please answer only in JSON.",
            }
        ],
        # And then pass the response_model = pydantic object you created
        response_model=User,
        max_retries=10
    )
    print(resp)

# Call the function 5 times
for _ in range(5):
    call_api()

name='DR. TAKASHI YAMADA' age=45 children=None fact=['Specializes in Internal Medicine', 'Speaks English, Japanese, and Mandarin', 'Author of several medical books']
name='DR. KENJI NARIYAMA' age=45 children=None fact=['Board-certified in Cardiology from Tokyo University Hospital', 'Fluent in Japanese and basic English', 'Skilled in traditional martial arts']
name='NAKAMURA TARO' age=45 children=None fact=['Specializes in cardiology', 'Graduated from Tokyo University Medical School', 'Spoken languages: Japanese and English']
name='DR. TARO YAMADA' age=35 children=None fact=['Speaks fluent English, Japanese, and Korean', 'Expertise in traditional Japanese medicine with a modern twist', 'Holder of the highest certification in acupuncture']
name='DR. TARO YAMADA' age=45 children=None fact=['Graduated from Tokyo University Medical School', 'Specialized in Internal Medicine and Cardiology', 'Published research on hypertension and cardiovascular disease prevention']


You can also get more information regarding the errors by using library Tenacity on top of pydaqntic

In [71]:
from pydantic import BaseModel, field_validator
import instructor
from instructor.exceptions import InstructorRetryException
from tenacity import Retrying, retry_if_not_exception_type, stop_after_attempt

# Patch the OpenAI client to enable response_model

class UserDetail(BaseModel):
    name: str
    age: int

    @field_validator("age")
    def validate_age(cls, v: int):
        raise ValueError(f"You will never succeed with {str(v)}")


retries = Retrying(
    retry=retry_if_not_exception_type(ZeroDivisionError), stop=stop_after_attempt(3)
)
# Use the client to create a user detail
try:
    user = client.chat.completions.create(
        model="llama3.1",
        response_model=UserDetail,
        messages=[{"role": "user", "content": "Extract Jason is 25 years old"}],
        max_retries=retries,
    )
except InstructorRetryException as e:
    print(e.messages[-1]["content"])  # type: ignore
    print(e.n_attempts)
    print(e.last_completion)

Recall the function correctly, fix the errors, exceptions found
1 validation error for UserDetail
age
  Value error, You will never succeed with 25 [type=value_error, input_value=25, input_type=int]
    For further information visit https://errors.pydantic.dev/2.9/v/value_error
3
ChatCompletion(id='chatcmpl-651', choices=[Choice(finish_reason=None, index=0, logprobs=None, message=ChatCompletionMessage(content='{"name": "Jason", "age": 25} \n\n \n\n \n \n\n \n\n \n \n  \n  \n\n\n    \n\n\n\n\n  \n\n\n   \n \n   \n\n  \n  \n\n\n \n \n \n\n \n\n \n\n\n \n\n\n\n\n\n\n  \n\n \n\n\n\n  \n\n \n\n\n\n\n \n\n \n\n\n  \n \n\n\n\n\n\n', refusal=None, role='assistant', function_call=None, tool_calls=None))], created=1727165969, model='llama3.1', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=13, prompt_tokens=149, total_tokens=162, completion_tokens_details=None))


LLM RAG

In [3]:
content = """The percentage of drug-treated represents the proportion of patients who receive cancer-specific pharmacotherapy outside clinical trial protocols. In our market, we cover only those therapies that thought leaders consider to be antineoplastic in their mechanism of action. We, therefore, do not cover supportive care strategies. We apply country-specific treatment rates to the various drug-treatable populations, thus forming our drug-treated populations.
We used the following sources to estimate the percentage of incident cases in the diagnosed population that are drug-treated:
• Opinions of thought leaders. We conducted in-depth interviews with 19 medical oncologists or urologists throughout the major markets who shared with us their insights into the percentage of patients who are drug-treated, treatment patterns, and medical practices. See “Methodology” for details.
• Physician surveys. We considered findings from 30 physicians surveyed by Clarivate to obtain information about drug treatment and medical practice in RCC. See “Methodology” for more information.
• Clarivate market model. Our estimates of the percentage of drug-treatable cases that are drug-treated are used in our patient-based market model for RCC and cross-checked against published sales data (where available) provided by company annual reports, SEC filings, and government health authorities in the first year of the study period. We reconcile our patient-based market model with our sales-based market model by adjusting variables (e.g., drug-treated patients, percentage treated with a particular drug, compliant cycles of therapy), as appropriate, to yield a validated base-year market model. On that basis, we forecast sales over a 10-year period.
We apply country-specific treatment rates to the five RCC populations covered in our study. The drug-treated populations do not include patients who are ineligible for antineoplastic drug therapy because of age, comorbidities, and/or poor PS; these patients can receive BSC only. The drug-treated populations also exclude drug treatment as part of a clinical trial protocol.
Early stage (stage I-III)
The early-stage (stage I-III) population comprises all newly diagnosed incident cases of stage I-III disease. In November 2017, Sutent became the first drug approved for the adjuvant treatment of RCC patients at high risk of recurrence following nephrectomy in the United States; as a result, drug-treatment rates in the United States increased gradually, as this agent provided a viable treatment option for patients who would otherwise not receive any drug treatment. Subsequently, Keytruda was approved in the major markets as an adjuvant treatment for RCC patients at intermediate-high or high risk of recurrence following nephrectomy. Welireg was approved in the United States and the United Kingdom as an adjuvant for VHL disease-associated RCC, a condition affecting approximately 6% of early-stage RCC patients. We estimate that approximately 30% of early-stage RCC patients are deemed to be at intermediate-high or high risk of recurrence, and are, therefore, eligible to receive adjuvant therapy. Based on our primary research, we estimate that the percentage of early-stage patients who are drug-treated is 10% in the United States, 1-4% in Europe, and 3% in Japan.
Following Keytruda’s approval, drug-treatment rates have been increasing substantially based on the agent’s DFS benefit versus placebo demonstrated in the KEYNOTE-564 trial. Despite the failures of several Phase 3 trials evaluating immune checkpoint inhibitors in the adjuvant setting (CheckMate-914, PROSPER, and IMmotion-010 trials), interviewed KOLs point out that Keytruda has clearly demonstrated efficacy in this setting and, therefore, the drug-treated population is expected to continue growing. Furthermore, we anticipate the approval of Welireg plus Keytruda during the forecast period; therefore, we expect that drug-treatment rates for early-stage disease will increase to 23% in the United States, 20-22% in Europe, and 20% in Japan in 2032.
First-line advanced or metastatic
The first-line treatable population comprises both newly diagnosed incident cases of patients with stage IV RCC and patients initially diagnosed with stage I-III disease that has recurred to stage IV disease. Based on our primary market research, we estimate drug-treatment rates of 76-85% in 2022 in the major markets we cover, and we forecast that these rates will increase slightly throughout the forecast period; from 2023 onward, novel regimens will enter the first-line market boosting drug-treatment rates. The drug-treated population does not include patients who are ineligible for antineoplastic drug therapy because of age, comorbidities, and/or poor PS; these patients may receive BSC only or refuse treatment. The drug-treated population also excludes drug treatment as part of a clinical trial protocol.
Second-line advanced or metastatic
RENAL CELL CARCINOMA DISEASE LANDSCAPE & FORECAST
clarivate.com © 2024 Clarivate. All rights reserved. 58
The second-line advanced or metastatic RCC drug-treatable population comprises patients who have progressed during or after receiving first-line drug treatment; we estimate that 80% of first-line drug-treated patients will become eligible for a second line of therapy. To generate the second-line RCC drug-treatable population, we use clinical trial information to calculate the percentage of patients still alive at the median time to progression after first-line treatment.
Overall, 71-80% of drug-treatable patients in the seven markets under study receive drug treatment in this setting. This rate will increase slightly over the forecast period, accounting for the entry of novel regimens (e.g., Opdivo plus Fotivda, Welireg plus Lenvima / Kisplyx) but will vary according to geographical nuances in the treatment practices for RCC.
Third- and fourth-line advanced or metastatic
Similar to the second-line advanced or metastatic drug-treatable population, the third- and fourth-line advanced or metastatic RCC drug-treatable populations comprise those patients who have progressed during or after receiving second- and third-line drug treatment, respectively; we estimate that 80% of second-line drug-treated patients will become eligible for a third line of therapy and that 70% of third-line patients will become eligible for a fourth line of therapy.
Overall, in 2022, 51-66% of patients in the seven markets under study received drug treatment in the third-line setting, and 29-40% of patients received a fourth line of therapy. However, we anticipate that drug-treatment rates in the third and fourth lines will increase during our forecast period as a result of the emergence of new therapies for previously untreated patients that will relegate current treatment options to later lines, increasing the number of treatment options available. In addition, we anticipate the entry of novel therapies in the third- and fourth-line settings that will boost drug-treatment rates. Although drug-treatment rates will increase in both the third and fourth lines (e.g., Welireg with or without Lenvima / Kisplyx), the increase in the fourth-line advanced or metastatic population will be greater than in the third line as options become available for patients who would otherwise receive no treatment outside of a clinical trial protocol or would receive BSC only."""

In [25]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
import instructor

class RCCTreatmentRates(BaseModel):
    first_line: int
    second_line: int | None # Optional[int] = None
    third_and_fourth_line: int

# Then Patch the client you want to use
client = instructor.from_openai(
    OpenAI(
        base_url="",
        api_key="ollama",  # required, but unused
    ),
#For Ollama you need to specify the mode argument
    mode=instructor.Mode.JSON,
)

resp = client.chat.completions.create(
    model="llama3.1",
    messages=[
        {
            "role": "user",
            "content": f" Retrieve information regarding the percentage of patient in each lines: first line, second line and third-fourth line for advanced or metastatic RCC drug-treatable population. Only get the number. Only use the information from the following context: {content}",
        }
    ],
# And then pass the response_model = pydantic object you created
    response_model=RCCTreatmentRates,
    max_retries=10

)

print(resp.model_dump_json(indent=2))

{
  "first_line": 76,
  "second_line": null,
  "third_and_fourth_line": 0
}
